## Messages

Messages are the basic unit for models in LangChain.
They represent the input and output of models.
They carry both content and metadata needed to represent the state of conversation when interacting with an LLM.

Message object contains : 
- Role - identifies the message type(e.g, system, user)
- Content - Represents the actual content of message (like text, images, audio, documents,etc.)
- Metadata - Optional fields such as response information, message IDs , and token usage.

LangChain Provides standard message type that works across all model providers, ensuring consistent behavious regardless of the model being called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")

### Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation history.

Internally, they are treated as Human message.
Here is an example.

In [3]:
model.invoke("Who directed Bahubali?")

AIMessage(content='The epic Indian film *Baahubali* was directed by **S.\u202fS.\u202fRajamouli**.', additional_kwargs={'reasoning_content': 'The user asks: "Who directed Bahubali?" They likely refer to the film "Baahubali" (two parts). The director is S. S. Rajamouli. So answer: It was directed by S. S. Rajamouli. Provide some context maybe. The user just asked. So answer simply.\n\nBut we must comply with policy. It\'s straightforward. No disallowed content. So answer.'}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 77, 'total_tokens': 197, 'completion_time': 0.130186089, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.005148703, 'prompt_tokens_details': None, 'queue_time': 0.316009383, 'total_time': 0.135334792}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16e4e-a5b9-76b3-9f05-b1fbd41

Use prompts when:
- You have single standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompts

Other type of prompt is message objects. They are of following types: 

- System message : Tells the model how to behave and provide context for interactions.
- Human Message : Represents user input and interactions with model
- AI message : Responses generated by model, including text context, tool calls, and metadata
- Tool message : Represents the outputs of tool calls

In [11]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

#### System Message

A System Message represent an initial set of instructions that primes the model's behaviour. You can use a system message to set the tine, define the model's role, and establish guidelines for responses.

#### Human Message

A Human Message represents user input and interactions. They can contain text, images, audio, files, and any other amount of multi-model content.

#### AI Message

An AI message represents the output of a model invocation. They can include multimodel data , tool calls, and provider specific metadata that you can later access.

#### Tool Message

For Models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.


================================================================

### Example 1: SystemMessage + HumanMessage
Set the model's role with a system message, then ask a question.

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage("You are a concise math tutor. Answer in 1-2 lines only."),
    HumanMessage("What is 15% of 200?"),
]

response = model.invoke(messages)
print(response.content)

15% of 200 is \(0.15 \times 200 = 30\).


### Example 2: SystemMessage + HumanMessage + AIMessage + HumanMessage
Pass conversation history manually to maintain context across turns.

In [14]:
messages = [
    SystemMessage("You are a helpful customer support agent for a laptop company."),
    HumanMessage("My laptop is not turning on."),
    AIMessage("Try holding the power button for 10 seconds, then release and press again."),
    HumanMessage("I tried that, still not working."),
]

response = model.invoke(messages)
print(response.content)

I’m sorry you’re still having trouble. Let’s go through a few more troubleshooting steps to pinpoint what’s going on.  

---

## 1. Quick sanity checks

| # | Check | How to do it | What to look for |
|---|-------|--------------|------------------|
| 1 | **Power adapter & cable** | Make sure the charger is plugged into a working outlet, and the charger’s LED (if it has one) is lit. | LED is off or flickering → try a different outlet or charger. |
| 2 | **Cable condition** | Inspect the power cord for kinks, frays, or any visible damage. | Damaged cable → replace it. |
| 3 | **Connection to laptop** | Ensure the adapter’s plug is fully seated in the laptop’s charging port. | Loose connection → firm it in. |
| 4 | **Battery indicator** | Look for a battery icon or LED on the laptop (some models light up when charging). | No light → the laptop may not be receiving power. |

---

## 2. Try a **hard reset**

1. **Disconnect everything** – unplug the charger, remove the battery (if removable

### Example 3: SystemMessage + HumanMessage + AIMessage (ToolCall) + ToolMessage + Final Response
Full tool execution loop — shows all 4 message types working together.

In [16]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol."""
    prices = {"AAPL": "$189.50", "GOOGL": "$175.20", "TSLA": "$245.80"}
    return prices.get(ticker.upper(), f"Ticker '{ticker}' not found.")

model_with_tools = model.bind_tools([get_stock_price])

# Step 1: System + Human → model decides to call a tool (AIMessage with tool_calls)
messages = [
    SystemMessage("You are a financial assistant."),
    HumanMessage("What is the current price of Apple stock?"),
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print("AI decided to call:", ai_msg.tool_calls)

# Step 2: Execute the tool → produces a ToolMessage
for tool_call in ai_msg.tool_calls:
    tool_result = get_stock_price.invoke(tool_call)
    messages.append(tool_result)
    print("ToolMessage:", tool_result)

# Step 3: Pass everything back → model gives final answer
final = model_with_tools.invoke(messages)
print("\nFinal Answer:", final.content)

AI decided to call: [{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'fc_882a2a2e-33b6-43bf-9551-cffc602754cc', 'type': 'tool_call'}]
ToolMessage: content='$189.50' name='get_stock_price' tool_call_id='fc_882a2a2e-33b6-43bf-9551-cffc602754cc'

Final Answer: Apple’s current stock price is **$189.50** per share.


In [19]:
final.usage_metadata

{'input_tokens': 175,
 'output_tokens': 33,
 'total_tokens': 208,
 'output_token_details': {'reasoning': 9}}

### Example 4: Multiple Tools + Multi-turn Conversation
Model picks the right tool based on the user's question.

In [17]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The weather in {city} is sunny and 25°C."

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    rates = {"USD_INR": 83.5, "INR_USD": 0.012, "USD_EUR": 0.92}
    key = f"{from_currency.upper()}_{to_currency.upper()}"
    rate = rates.get(key)
    if rate:
        return f"{amount} {from_currency} = {amount * rate:.2f} {to_currency}"
    return "Conversion rate not available."

model_multi = model.bind_tools([get_weather, convert_currency])

def run_tool_loop(user_message: str):
    """Runs a full tool loop for one user message."""
    messages = [SystemMessage("You are a helpful assistant."), HumanMessage(user_message)]
    ai_msg = model_multi.invoke(messages)
    messages.append(ai_msg)
    tool_map = {"get_weather": get_weather, "convert_currency": convert_currency}
    for tool_call in ai_msg.tool_calls:
        result = tool_map[tool_call["name"]].invoke(tool_call)
        messages.append(result)
    final = model_multi.invoke(messages)
    return final.content

# Turn 1: triggers get_weather
print("Turn 1:", run_tool_loop("What's the weather like in Tokyo?"))

# Turn 2: triggers convert_currency
print("\nTurn 2:", run_tool_loop("Convert 100 USD to INR."))

Turn 1: It’s sunny in Tokyo right now, with the temperature around 25 °C.

Turn 2: 100 USD = 8,350.00 INR (using the current exchange rate).
